# 12 — Refusal Direction on Llama 3.3 70B (Arditi pipeline)

Self-contained notebook that runs the full refusal-direction extraction pipeline from Arditi et al. *Refusal in Language Models Is Mediated by a Single Direction* (arXiv 2406.11717) on **`meta-llama/Llama-3.3-70B-Instruct`**.

**Repo:** https://github.com/andyrdt/refusal_direction — already cloned at `third_party/refusal_direction/`. The `Llama3Model` class in `pipeline/model_utils/llama3_model.py` matches any path containing `'llama-3'` (lowercased), so 3.3-70B drops in.

**What the pipeline does** (reproduces paper §3 + §4):
1. Sample 128 harmful + 128 harmless training instructions, 32 + 32 val.
2. Filter by current-model refusal score so train/val arms are clean.
3. Generate per-layer mean-difference candidate directions (harmful − harmless residual stream).
4. Score each candidate by ablation effect (drop in refusal score on harmful val) + side-effect on harmless val + KL.
5. Save the winning `direction.pt`, `pos`, `layer`.
6. Generate completions on JailbreakBench under three conditions (baseline / direction-ablated / actadd-with-+1×direction-on-harmless), evaluate with substring matching + LlamaGuard 2.
7. Evaluate CE-loss change on harmless distribution.

**Outputs (saved to `OUT_DIR`):**
- `direction.pt` — the canonical refusal direction `(d_model,)`.
- `direction_metadata.json` — `{pos, layer}`.
- `generate_directions/mean_diffs.pt` — all candidate directions, shape `(n_pos, n_layers, d_model)`.
- `select_direction/*.json` — per-candidate refusal/KL scores.
- `completions/*.json` — JailbreakBench completions and evals.
- `loss_evals/*.json` — CE-loss numbers.

**Compute:** Llama 3.3 70B in bf16 needs ~140 GB. Recommended: **4× A100 80 GB** or **2× H100 80 GB** with `device_map='auto'`. Pipeline does many forward passes (direction generation + selection sweep + JailbreakBench gen across 3 conditions + 2048 CE-loss batches). Expect **6–12 hours** wall-clock.

**Tokens needed:**
- `HF_TOKEN` — Llama 3.3 is gated.
- `TOGETHER_API_KEY` — used by `evaluate_jailbreak` for LlamaGuard 2 scoring. If you skip this, swap `jailbreak_eval_methodologies` to substring-only in the config cell.

**Resumability:** The pipeline writes intermediate artifacts as it goes. If a stage finishes, re-running skips re-doing it as long as the directory exists. To restart from clean, delete `OUT_DIR`.

**Why a notebook (not a shell script):** lets us patch token, override config, and inspect intermediates between stages. The actual heavy lifting is `pipeline.run_pipeline.run_pipeline(model_path)` — most cells below are setup.

## 0 — Sanity: GPU check

Bail early if no CUDA or insufficient VRAM.

In [ ]:
import subprocess, torch
print(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), 'no CUDA — refusal_direction needs CUDA (vllm + flash-attn)'
n = torch.cuda.device_count()
total_gb = sum(torch.cuda.get_device_properties(i).total_memory for i in range(n)) / 1e9
print(f'{n} GPUs visible, total VRAM = {total_gb:.0f} GB')
assert total_gb >= 140, 'Llama 3.3 70B in bf16 needs ~140 GB total VRAM'

## 1 — Install dependencies

Two paths: (a) the repo is already on disk (you rsynced the project) — just install requirements; (b) fresh box — clone + install. Note: `requirements.txt` pins old versions (`torch==2.3.0`, `vllm==0.5.0`, `transformers==4.44.2`). Llama 3.3 was released with these wheels current; if you're on a newer driver / CUDA you may need to bump `torch` and refetch `vllm`/`flash-attn` matching wheels. Try the pinned versions first.

In [ ]:
import os
from pathlib import Path

# Adjust if you rsynced the project elsewhere.
PROJECT_ROOT = Path(os.environ.get('MECH_SPOOF_ROOT', '/workspace/Mech_spoof'))
REPO_DIR = PROJECT_ROOT / 'third_party' / 'refusal_direction'

if not REPO_DIR.exists():
    print('cloning fresh into /tmp/refusal_direction')
    REPO_DIR = Path('/tmp/refusal_direction')
    if not REPO_DIR.exists():
        os.system(f'git clone https://github.com/andyrdt/refusal_direction.git {REPO_DIR}')

print('repo at', REPO_DIR)
assert (REPO_DIR / 'pipeline' / 'run_pipeline.py').exists()

In [ ]:
# Install requirements. Skip if already installed.
# If wheels fail (CUDA mismatch), comment out vllm / xformers and retry — most of the
# pipeline only uses HF transformers; vllm is used by evaluate_jailbreak's generation path.
!pip install -q -r {REPO_DIR}/requirements.txt

## 2 — Tokens + working dir

**Do NOT run `setup.sh`** from the cloned repo — its first line is `echo "" > .env` which would clobber the project's existing `.env`. Set tokens directly here instead.

In [ ]:
try:
    from google.colab import drive, userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    try:
        os.environ['TOGETHER_API_KEY'] = userdata.get('TOGETHER_API_KEY')
    except Exception:
        print('no TOGETHER_API_KEY in userdata — LlamaGuard 2 eval will fail; will fall back below')
    drive.mount('/content/drive')
    DRIVE_ROOT = '/content/drive/MyDrive/mech_spoof_results'
except Exception:
    DRIVE_ROOT = os.environ.get('DRIVE_ROOT', str(PROJECT_ROOT / 'refusal_direction_runs'))

assert os.environ.get('HF_TOKEN'), 'HF_TOKEN must be set (Llama 3.3 is gated)'
have_together = bool(os.environ.get('TOGETHER_API_KEY'))
print('HF_TOKEN          :', 'set' if os.environ.get('HF_TOKEN') else 'MISSING')
print('TOGETHER_API_KEY  :', 'set' if have_together else 'missing (will use substring-only eval)')
print('DRIVE_ROOT        :', DRIVE_ROOT)

# HF login (so AutoTokenizer/Model can fetch gated weights).
from huggingface_hub import login
login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)

In [ ]:
# Make pipeline imports work — they expect to run from the repo root.
import sys
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
print('cwd:', os.getcwd())

## 3 — Config patches

Default `Config` puts artifacts under `pipeline/runs/{model_alias}/` inside the repo. We override `artifact_path` to point at Drive (or wherever `DRIVE_ROOT` is) so artifacts survive the box being torn down.

Also: if `TOGETHER_API_KEY` is missing, drop `llamaguard2` from `jailbreak_eval_methodologies`.

In [ ]:
from pipeline.config import Config

MODEL_PATH  = 'meta-llama/Llama-3.3-70B-Instruct'
MODEL_ALIAS = MODEL_PATH.split('/')[-1].lower()  # 'llama-3.3-70b-instruct'
OUT_DIR     = Path(DRIVE_ROOT) / 'refusal_direction' / MODEL_ALIAS
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Patch Config.artifact_path to redirect into OUT_DIR.
_orig_artifact_path = Config.artifact_path
def _patched(self):
    return str(OUT_DIR)
Config.artifact_path = _patched

# If no Together key, skip LlamaGuard. (substring matching still runs.)
if not have_together:
    Config.__dataclass_fields__['jailbreak_eval_methodologies'].default = ('substring_matching',)

cfg = Config(model_alias=MODEL_ALIAS, model_path=MODEL_PATH)
print('artifact_path:', cfg.artifact_path())
print('jailbreak eval:', cfg.jailbreak_eval_methodologies)
print('refusal eval  :', cfg.refusal_eval_methodologies)
print('n_train/n_val/n_test:', cfg.n_train, cfg.n_val, cfg.n_test)
print('eval datasets:', cfg.evaluation_datasets)

## 4 — Run the pipeline

Calls `run_pipeline.run_pipeline(model_path)` directly so we avoid argparse / CLI invocation. This is the long part. The 70B model loads once, then steps 1–5 reuse it.

If it crashes mid-way, intermediate `.pt` / `.json` artifacts in `OUT_DIR` will let you skip re-doing earlier stages. Re-running the cell picks up where it left off (the pipeline checks `os.path.exists` on each step's output dir).

In [ ]:
from pipeline.run_pipeline import run_pipeline
import time

t0 = time.time()
run_pipeline(model_path=MODEL_PATH)
print(f'\npipeline complete in {(time.time() - t0) / 3600:.2f} h')

## 5 — Inspect the result

Load the selected direction + metadata. Expected: `pos` is one of {-1, -2, -3, -4, -5} (last-few-token positions of the `[/INST]`-equivalent EOI tokens), `layer` somewhere in the middle of the network (~1/3 to 1/2 depth).

In [ ]:
import json, torch

direction = torch.load(OUT_DIR / 'direction.pt')
metadata  = json.loads((OUT_DIR / 'direction_metadata.json').read_text())
print('selected pos  :', metadata['pos'])
print('selected layer:', metadata['layer'])
print('direction     :', tuple(direction.shape), direction.dtype)
print('||direction|| :', float(direction.norm()))

# Show per-candidate scores from the selection sweep.
select_dir = OUT_DIR / 'select_direction'
for f in sorted(select_dir.iterdir()):
    print('  ', f.name, '  ', f.stat().st_size, 'bytes')

In [ ]:
# Show evaluation summary across {baseline, ablation, actadd}.
for dataset in cfg.evaluation_datasets + ('harmless',):
    print(f'\n=== {dataset} ===')
    for label in ('baseline', 'ablation', 'actadd'):
        p = OUT_DIR / 'completions' / f'{dataset}_{label}_evaluations.json'
        if not p.exists():
            continue
        ev = json.loads(p.read_text())
        print(f'  {label:9s}', {k: v for k, v in ev.items() if k != 'completions'})

In [ ]:
# CE-loss summary.
for label in ('baseline', 'ablation', 'actadd'):
    p = OUT_DIR / 'loss_evals' / f'{label}_loss_eval.json'
    if p.exists():
        print(f'{label:9s}', json.loads(p.read_text()))

## 6 — Wrap up

Final artifact tree in `OUT_DIR`. Copy this whole directory back to the project (`Mech_spoof/exp_attacks/` or similar) so the rest of the codebase can use the direction.

In [ ]:
for root, dirs, files in os.walk(OUT_DIR):
    rel = Path(root).relative_to(OUT_DIR)
    for f in files:
        size = (Path(root) / f).stat().st_size
        print(f'  {rel}/{f}'.lstrip('/'), f'  {size:>10} B')

## Notes & gotchas

- **Refusal-token list.** `Llama3Model._get_refusal_toks()` returns `[40]` (token `'I'`). The selection step scores candidates by drop in P(`'I'`) on harmful prompts. For Llama 3.3 the tokenizer is the same as Llama 3 / 3.1, so this token id is still valid — but if the result looks weird, double-check `tokenizer.encode('I', add_special_tokens=False) == [40]`.
- **Chat template.** `LLAMA3_CHAT_TEMPLATE` in `llama3_model.py` is the Llama 3 format with `<|begin_of_text|>`, `<|start_header_id|>`, etc. Llama 3.3 uses the same template, so this works as-is.
- **vllm version.** `vllm==0.5.0` predates Llama 3.3 by a lot. If `evaluate_jailbreak` blows up trying to load the 70B with vllm 0.5.0, two options: (a) bump to a recent vllm that supports Llama 3.3, (b) skip vllm entirely by setting `jailbreak_eval_methodologies = ('substring_matching',)` (we already do this if no Together key) — substring eval uses HF generation, not vllm.
- **Memory headroom for selection.** The selection step holds the full `(n_pos, n_layers, d_model)` candidate tensor *plus* runs forward passes with each candidate ablated. On 70B that's `5 × 80 × 8192 × 4 B = ~13 MB` for the candidates themselves (negligible) but the per-candidate forward sweeps add up. If OOM, lower `n_train` / `n_val` first.
- **`pos = -1` vs `pos = -5`.** The pipeline scores candidates from the last 5 tokens of the End-Of-Instruction segment. Llama-3-8B selected `pos=-5, layer=12`. For 70B expect a deeper layer, ~25–35.
- **Reproducing across runs.** Random seed is fixed (42) inside `load_and_sample_datasets`, so re-runs on the same model produce the same `direction.pt`.